# CycloneX: Tropical Cyclone Pattern Identification, Intensity Estimation & Track Forecasting
### SIH Problem Statement 26070 (Ministry of Earth Sciences / IMD)

> **DISCLAIMER**: This notebook is an AI-assisted research prototype for decision support. It is not an official IMD warning.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q numpy pandas scipy scikit-learn h5py pillow matplotlib requests httpx fastapi uvicorn

## 2. Generate / Load TCIR Dataset & Preprocessing
Loads multi-channel satellite data (IR1, WV, PMW), computes robust normalization on train split only, and partitions by unique cyclone ID.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../data')
sys.path.insert(0, '../models')
sys.path.insert(0, '../api')

from download_tcir import load_raw_tcir, create_sample_tcir_dataset
from preprocess import preprocess_dataset

# Ensure sample dataset is ready
sample_h5 = '../data/tcir_sample.h5'
if not os.path.exists(sample_h5):
    create_sample_tcir_dataset(sample_h5, num_cyclones=10)

raw_images, meta_df = load_raw_tcir(sample_h5)
prep = preprocess_dataset(raw_images, meta_df)
print(f"Preprocessed Images Shape: {prep['images'].shape}")
print(f"Train frames: {len(prep['splits']['train_indices'])}, Val: {len(prep['splits']['val_indices'])}, Test: {len(prep['splits']['test_indices'])}")

## 3. Visualizing Satellite Channels (IR1, WV, PMW)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
frame = prep['images'][0]

axes[0].imshow(frame[:, :, 0], cmap='inferno')
axes[0].set_title('IR1 (Infrared Channel)')

axes[1].imshow(frame[:, :, 1], cmap='viridis')
axes[1].set_title('WV (Water Vapor)')

axes[2].imshow(frame[:, :, 2], cmap='plasma')
axes[2].set_title('PMW (Passive Microwave)')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Grad-CAM Visual Explainability
Calculates activation gradients to identify which convective cloud bands drive the model's intensity estimation.

In [ ]:
from gradcam import generate_gradcam_artifact

gc_res = generate_gradcam_artifact(frame, model=None, alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(frame[:, :, 0], cmap='gray')
axes[0].set_title('IR1 Brightness Temp')

axes[1].imshow(gc_res['heatmap'], cmap='jet')
axes[1].set_title('Grad-CAM Heatmap')

axes[2].imshow(gc_res['blended_rgb'])
axes[2].set_title('Overlaid Explainability')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Temporal Track Forecasting & Persistence Comparison

In [ ]:
from train_track_model import run_track_training_comparison

perf_report = run_track_training_comparison(
    csv_path='../data/sample_best_tracks.csv',
    metrics_output_path='../models/track_model_performance.json'
)
print("Model Comparison Results:")
print(f"Persistence 24h Mean Error: {perf_report['persistence_baseline']['haversine_track_error_km']['24h']['mean_haversine_km']} km")
print(f"GRU Forecaster 24h Mean Error: {perf_report['gru_model']['haversine_track_error_km']['24h']['mean_haversine_km']} km")
print(f"Improvement vs Persistence: {perf_report['summary']['error_reduction_24h_pct']}%")

## 6. Querying the FastAPI Backend

In [ ]:
from risk import evaluate_cyclone_risk

risk_res = evaluate_cyclone_risk(
    cyclone_id='FANI_2019',
    current_vmax_kt=115.0,
    current_lat=14.8,
    current_lon=84.2,
    forecast_vmax_24h_kt=120.0,
    distance_to_coastline_km=85.0
)

print(f"Cyclone Fani Risk Level: {risk_res.overall_risk_level} ({risk_res.total_risk_score}/100)")
print(f"Proximity Threat: {risk_res.proximity_threat}")
for r in risk_res.rule_audit_trace:
    if r.triggered:
        print(f"  [Triggered] {r.rule_id}: {r.condition} (+{r.risk_score_impact} pts)")